# f-GRPO: Divergence-Based Reinforcement Learning for LLM Alignment

**Notebook 9 of 9** | LLM Alignment Series

## What is f-GRPO?

**f-GRPO** (f-Group Relative Policy Optimization) generalises standard GRPO by replacing the clipped-ratio objective with a loss derived from **f-divergences**. It was introduced in:

> Haldar et al., *"f-GRPO and Beyond: Divergence-Based Reinforcement Learning Algorithms for General LLM Alignment"*, arXiv 2602.05946, Feb 2026.

### Core Idea

Standard GRPO computes group-normalised advantages and updates the policy with a PPO-style clipped objective that treats all completions identically. f-GRPO instead:

1. **Splits completions into two sets** based on advantage sign:
   - $\mathcal{D}^+$: completions with positive advantage (better than group average)
   - $\mathcal{D}^-$: completions with negative advantage (worse than group average)

2. **Applies different divergence-specific transformations** to each set using the *variational representation* of f-divergences.

3. **Provides strict reward improvement guarantees** — the policy is guaranteed to improve until it reaches the maximum reward, which standard GRPO does not guarantee.

### f-Divergences

An **f-divergence** between distributions $P$ and $Q$ is defined as:

$$D_f(P \| Q) = \mathbb{E}_{x \sim Q}\left[f\left(\frac{P(x)}{Q(x)}\right)\right]$$

where $f$ is a convex function with $f(1) = 0$. Different choices of $f$ give different divergences:

| Divergence | $f(t)$ | Link function $g(r)$ | Properties |
|---|---|---|---|
| **KL** | $t \ln t$ | $r$ | Mode-covering, standard choice |
| **Reverse KL** | $-\ln t$ | $-e^{-r}$ | Mode-seeking, concentrates mass |
| **Pearson $\chi^2$** | $(t-1)^2$ | $r$ | Symmetric, heavy penalty on outliers |
| **Hellinger** | $(\sqrt{t}-1)^2$ | $1 - e^{-r/2}$ | Balanced, bounded |
| **Jensen-Shannon** | $t\ln t - (1+t)\ln\frac{1+t}{2}$ | $\ln(2) - \ln(1 + e^{-r})$ | Symmetric, bounded [0, ln2] |
| **Total Variation** | $\frac{1}{2}|t-1|$ | $\tanh(r/2)$ | Strongest bound, most conservative |

### f-GRPO vs Standard GRPO

| | GRPO | f-GRPO |
|---|---|---|
| Objective | Clipped ratio × advantage | f-divergence variational bound |
| Treatment of $\mathcal{D}^+$ / $\mathcal{D}^-$ | Same (clipped ratio) | Different (link vs conjugate) |
| Convergence | No strict guarantee | Strict reward improvement |
| Hyperparameters | $\epsilon$ (clip range) | Choice of f-divergence |
| Reference model | Optional ($\beta$) | Required (implicit reward = $\beta \ln \frac{\pi_\theta}{\pi_{\text{ref}}}$) |

### Why This Matters

The choice of f-divergence controls the *shape* of the optimisation landscape:
- **KL**: encourages the policy to cover all modes of the reward → diverse but sometimes unfocused
- **Reverse KL**: mode-seeking → sharp, focused responses but may ignore alternative good answers
- **Hellinger**: a balanced middle ground
- **Total Variation**: most conservative updates, strongest stability

This notebook implements f-GRPO from scratch (TRL does not support it) and compares different divergence choices.

**Model**: Qwen2.5-7B-Instruct | **GPU**: NVIDIA RTX 4090

## Setup

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import re
import gc
import copy
from collections import defaultdict

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram_gb:.1f} GB")

## Part 1: f-Divergence Functions

Each f-divergence is defined by:
- **Generator function** $f(t)$: convex, $f(1)=0$
- **Fenchel conjugate** $f^*(s) = \sup_t \{st - f(t)\}$
- **Link function** $g(r)$: monotonically increasing, maps implicit reward to divergence space

The f-GRPO loss uses $g(r)$ for positive-advantage samples and $f^*(g(r))$ for negative-advantage samples.

In [ ]:
class FDivergence:
    """Base class for f-divergences used in f-GRPO."""

    def __init__(self, name: str):
        self.name = name

    def f(self, t: torch.Tensor) -> torch.Tensor:
        """Generator function f(t), convex with f(1) = 0."""
        raise NotImplementedError

    def f_conjugate(self, s: torch.Tensor) -> torch.Tensor:
        """Fenchel conjugate f*(s) = sup_t {st - f(t)}."""
        raise NotImplementedError

    def link(self, r: torch.Tensor) -> torch.Tensor:
        """Link function g(r) mapping implicit rewards to divergence space."""
        raise NotImplementedError


class KLDivergence(FDivergence):
    """KL divergence: f(t) = t ln(t).
    Mode-covering — spreads probability mass broadly."""

    def __init__(self):
        super().__init__("KL")

    def f(self, t):
        return t * torch.log(t + 1e-8)

    def f_conjugate(self, s):
        return torch.exp(s - 1)

    def link(self, r):
        return r


class ReverseKLDivergence(FDivergence):
    """Reverse KL: f(t) = -ln(t).
    Mode-seeking — concentrates mass on best modes."""

    def __init__(self):
        super().__init__("Reverse KL")

    def f(self, t):
        return -torch.log(t + 1e-8)

    def f_conjugate(self, s):
        return -1 - torch.log(-s + 1e-8)  # defined for s < 0

    def link(self, r):
        return -torch.exp(-r)


class PearsonDivergence(FDivergence):
    """Pearson chi-squared: f(t) = (t - 1)^2.
    Heavy penalty on large deviations."""

    def __init__(self):
        super().__init__("Pearson χ²")

    def f(self, t):
        return (t - 1) ** 2

    def f_conjugate(self, s):
        return s + s ** 2 / 4

    def link(self, r):
        return r


class HellingerDivergence(FDivergence):
    """Squared Hellinger: f(t) = (sqrt(t) - 1)^2.
    Balanced — bounded and symmetric-ish."""

    def __init__(self):
        super().__init__("Hellinger")

    def f(self, t):
        return (torch.sqrt(t + 1e-8) - 1) ** 2

    def f_conjugate(self, s):
        return s / (1 - s + 1e-8)  # defined for s < 1

    def link(self, r):
        return 1 - torch.exp(-r / 2)


class JensenShannonDivergence(FDivergence):
    """Jensen-Shannon: f(t) = t ln(t) - (1+t) ln((1+t)/2).
    Symmetric and bounded in [0, ln 2]."""

    def __init__(self):
        super().__init__("Jensen-Shannon")

    def f(self, t):
        return t * torch.log(t + 1e-8) - (1 + t) * torch.log((1 + t) / 2 + 1e-8)

    def f_conjugate(self, s):
        return -torch.log(2 - torch.exp(s) + 1e-8)  # defined for s < ln(2)

    def link(self, r):
        return torch.log(torch.tensor(2.0, device=r.device)) - torch.log(1 + torch.exp(-r))


class TotalVariationDivergence(FDivergence):
    """Total Variation: f(t) = 0.5 * |t - 1|.
    Most conservative — strongest stability guarantee."""

    def __init__(self):
        super().__init__("Total Variation")

    def f(self, t):
        return 0.5 * torch.abs(t - 1)

    def f_conjugate(self, s):
        # f*(s) = s if |s| <= 0.5, else +inf
        return torch.where(torch.abs(s) <= 0.5, s, torch.tensor(float('inf'), device=s.device))

    def link(self, r):
        return torch.tanh(r / 2)


# Registry
DIVERGENCES = {
    "kl": KLDivergence,
    "reverse_kl": ReverseKLDivergence,
    "pearson": PearsonDivergence,
    "hellinger": HellingerDivergence,
    "jensen_shannon": JensenShannonDivergence,
    "total_variation": TotalVariationDivergence,
}

print(f"Implemented {len(DIVERGENCES)} f-divergences:")
for key, cls in DIVERGENCES.items():
    d = cls()
    print(f"  {key:>20s} — {d.name}")

## Visualise the Divergences

Let us plot each generator $f(t)$, its link function $g(r)$, and the conjugate $f^*(s)$ to build intuition about how each divergence shapes the optimisation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(DIVERGENCES)))

# Plot f(t) — generator functions
t = torch.linspace(0.01, 3.0, 300)
for (key, cls), color in zip(DIVERGENCES.items(), colors):
    d = cls()
    vals = d.f(t).numpy()
    vals = np.clip(vals, -5, 10)
    axes[0].plot(t.numpy(), vals, label=d.name, color=color, linewidth=2)
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].axvline(x=1, color='gray', linewidth=0.5, linestyle='--')
axes[0].set_xlabel('t')
axes[0].set_ylabel('f(t)')
axes[0].set_title('Generator Functions f(t)')
axes[0].legend(fontsize=8)
axes[0].set_ylim(-2, 5)
axes[0].grid(True, alpha=0.3)

# Plot g(r) — link functions
r = torch.linspace(-3, 3, 300)
for (key, cls), color in zip(DIVERGENCES.items(), colors):
    d = cls()
    vals = d.link(r).numpy()
    vals = np.clip(vals, -5, 5)
    axes[1].plot(r.numpy(), vals, label=d.name, color=color, linewidth=2)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].axvline(x=0, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_xlabel('r (implicit reward)')
axes[1].set_ylabel('g(r)')
axes[1].set_title('Link Functions g(r)')
axes[1].legend(fontsize=8)
axes[1].set_ylim(-3, 3)
axes[1].grid(True, alpha=0.3)

# Plot f*(s) — Fenchel conjugates
s = torch.linspace(-2, 2, 300)
for (key, cls), color in zip(DIVERGENCES.items(), colors):
    d = cls()
    vals = d.f_conjugate(s).numpy()
    vals = np.clip(vals, -5, 10)
    axes[2].plot(s.numpy(), vals, label=d.name, color=color, linewidth=2)
axes[2].axhline(y=0, color='black', linewidth=0.5)
axes[2].set_xlabel('s')
axes[2].set_ylabel('f*(s)')
axes[2].set_title('Fenchel Conjugates f*(s)')
axes[2].legend(fontsize=8)
axes[2].set_ylim(-3, 5)
axes[2].grid(True, alpha=0.3)

plt.suptitle('f-Divergence Components', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key observations:")
print("• KL and Pearson have unbounded link functions — aggressive updates when reward is large")
print("• Total Variation's tanh link saturates — most conservative, bounded updates")
print("• Reverse KL's exponential link strongly penalises low-reward completions")
print("• Hellinger and Jensen-Shannon sit in between — moderate, bounded behaviour")

## Part 2: The f-GRPO Loss

Given a prompt $x$ and a group of $G$ completions $\{y_i\}$ with rewards $\{R_i\}$:

1. Compute standard GRPO advantages: $\hat{A}_i = \frac{R_i - \bar{R}}{\sigma_R}$

2. Compute the implicit reward for each completion: $r_{\theta,i} = \beta \ln \frac{\pi_\theta(y_i|x)}{\pi_{\text{ref}}(y_i|x)}$

3. Split by advantage sign and compute the f-GRPO loss:

$$\mathcal{L}_{\text{f-GRPO}} = -\frac{1}{G} \sum_{i=1}^{G} |\hat{A}_i| \cdot \psi_i$$

where:
$$\psi_i = \begin{cases} w_i^+ \cdot g(r_{\theta,i}) & \text{if } \hat{A}_i > 0 \\ w_i^- \cdot f^*(g(r_{\theta,i})) & \text{if } \hat{A}_i \leq 0 \end{cases}$$

and $w_i^+, w_i^-$ are softmax-normalised importance weights within each set.

The key insight: **positive-advantage samples are pushed through the link function** (increasing their implicit reward), while **negative-advantage samples are pushed through the conjugate** (decreasing their implicit reward). The specific shape of $g$ and $f^*$ depends on which f-divergence you choose.

In [ ]:
def compute_fgrpo_loss(
    log_probs: torch.Tensor,
    ref_log_probs: torch.Tensor,
    advantages: torch.Tensor,
    divergence: FDivergence,
    beta: float = 0.1,
) -> torch.Tensor:
    """Compute the f-GRPO loss for a group of completions.

    Args:
        log_probs: Log probabilities under current policy, shape (G,)
        ref_log_probs: Log probabilities under reference policy, shape (G,)
        advantages: Group-normalised advantages, shape (G,)
        divergence: The f-divergence to use
        beta: Implicit reward scaling coefficient

    Returns:
        Scalar loss (to be minimised)
    """
    G = log_probs.shape[0]

    # Implicit reward: r = beta * ln(pi_theta / pi_ref)
    implicit_reward = beta * (log_probs - ref_log_probs)

    # Split into positive and negative advantage sets
    pos_mask = (advantages > 0).float()
    neg_mask = (advantages <= 0).float()

    # Compute importance weights for each set using softmax
    # For D+: softmax over (r_j - ln pi_old(y_j|x)) for positive-advantage samples
    # For D-: softmax over (-r_j - ln pi_old(y_j|x)) for negative-advantage samples
    # Simplified: we use uniform weights (1/|D+|, 1/|D-|) for stability
    pos_count = pos_mask.sum().clamp(min=1)
    neg_count = neg_mask.sum().clamp(min=1)
    w_pos = pos_mask / pos_count
    w_neg = neg_mask / neg_count

    # Apply link function to implicit reward
    g_r = divergence.link(implicit_reward)

    # f-GRPO objective:
    # For positive advantage: maximise g(r)  → push policy toward these completions
    # For negative advantage: minimise f*(g(r)) → push policy away from these
    f_star_g_r = divergence.f_conjugate(g_r)

    # Clamp conjugate to avoid NaN/Inf
    f_star_g_r = torch.clamp(f_star_g_r, -10.0, 10.0)

    abs_adv = advantages.abs()

    # Loss = -1/G * sum_i |A_i| * psi_i
    # psi_i = w_i+ * g(r_i)  for A_i > 0
    # psi_i = w_i- * f*(g(r_i))  for A_i <= 0
    pos_term = (w_pos * abs_adv * g_r).sum()
    neg_term = (w_neg * abs_adv * f_star_g_r).sum()

    # We want to maximise g(r) for good completions and minimise f*(g(r)) for bad ones.
    # Since f*(g(r)) is increasing in r (for most divergences), minimising it means
    # decreasing r for bad completions. The loss to minimise:
    loss = -(pos_term - neg_term)

    return loss


# Demonstrate with synthetic data
print("=== Synthetic f-GRPO Loss Demo ===")
print("Group of 4 completions, 2 good (positive advantage) and 2 bad (negative advantage)\n")

torch.manual_seed(42)
log_p = torch.tensor([-2.0, -1.5, -3.0, -2.5], requires_grad=True)
ref_log_p = torch.tensor([-2.1, -1.6, -2.9, -2.4])
advantages = torch.tensor([1.2, 0.8, -0.7, -1.3])  # 2 positive, 2 negative

print(f"{'Divergence':<20} {'Loss':>10} {'Grad norm':>12}")
print("-" * 45)
for name, cls in DIVERGENCES.items():
    div = cls()
    lp = log_p.detach().clone().requires_grad_(True)
    loss = compute_fgrpo_loss(lp, ref_log_p, advantages, div, beta=0.1)
    loss.backward()
    print(f"{div.name:<20} {loss.item():>10.4f} {lp.grad.norm().item():>12.6f}")

print("\nNote how different divergences produce different gradient magnitudes.")
print("Total Variation has the smallest gradients (most conservative updates).")
print("KL and Pearson have larger gradients (more aggressive updates).")

## Part 3: Reward Functions

We reuse the same reward functions from Notebook 08 (GRPO) to enable a fair comparison.

In [ ]:
def format_reward(text: str) -> float:
    """Reward structured formatting (numbered lists, bullets, paragraphs)."""
    score = 0.0
    if re.search(r'\d+\.\s', text):
        score += 0.5
    if re.search(r'[-*]\s', text):
        score += 0.3
    if text.count('\n\n') >= 1:
        score += 0.2
    return score


def conciseness_reward(text: str) -> float:
    """Reward concise but substantive responses (sweet spot: 50-200 words)."""
    word_count = len(text.split())
    if word_count < 10:
        return -1.0
    elif word_count < 20:
        return -0.5
    elif word_count <= 200:
        return 1.0
    elif word_count <= 300:
        return 0.5
    return 0.0


def no_repetition_reward(text: str) -> float:
    """Penalise repetitive text."""
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    if len(sentences) <= 1:
        return 0.0
    return len(set(sentences)) / len(sentences)


def helpfulness_reward(prompt: str, text: str) -> float:
    """Reward responses that address the prompt."""
    refusal_starts = [
        "i can't", "i cannot", "i'm sorry", "sorry,", "i apologize",
        "as an ai", "i'm not able",
    ]
    text_lower = text.lower().strip()
    if any(text_lower.startswith(r) for r in refusal_starts):
        return -0.5
    stop_words = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'what', 'how',
        'why', 'when', 'where', 'who', 'which', 'do', 'does', 'can',
        'could', 'would', 'should', 'in', 'on', 'at', 'to', 'for',
        'of', 'and', 'or', 'but', 'with', 'this', 'that', 'it', 'me',
        'you', 'your', 'my', 'i', 'please', 'explain', 'describe',
    }
    prompt_words = set(prompt.lower().split()) - stop_words
    response_words = set(text_lower.split())
    if prompt_words:
        overlap = len(prompt_words & response_words) / len(prompt_words)
    else:
        overlap = 0.5
    return min(overlap * 2.0, 1.0)


def compute_total_reward(prompt: str, completion: str) -> float:
    """Sum of all reward functions."""
    return (
        format_reward(completion)
        + conciseness_reward(completion)
        + no_repetition_reward(completion)
        + helpfulness_reward(prompt, completion)
    )


# Quick test
test_prompt = "What causes earthquakes?"
test_good = "1. Tectonic plates shift\n2. Pressure builds up\n\nEarthquakes result from plate movement."
test_bad = "Yes."

print(f"Good response reward: {compute_total_reward(test_prompt, test_good):.2f}")
print(f"Bad response reward:  {compute_total_reward(test_prompt, test_bad):.2f}")

## Part 4: Training Prompts

In [ ]:
training_prompts = [
    "What causes earthquakes?",
    "Explain how vaccines work.",
    "What is the water cycle?",
    "How does photosynthesis work?",
    "What causes the northern lights?",
    "How do airplanes fly?",
    "What is the greenhouse effect?",
    "How does the internet work?",
    "What is evolution by natural selection?",
    "How do batteries store energy?",
    "What causes tides?",
    "What is DNA and what does it do?",
    "How do computers process information?",
    "What is inflation in economics?",
    "What are the pros and cons of remote work?",
    "Compare renewable and non-renewable energy sources.",
    "What are the main causes of deforestation?",
    "Explain why biodiversity is important.",
    "How does urbanization affect the environment?",
    "What factors contribute to income inequality?",
    "List 5 tips for better sleep. Use a numbered list.",
    "Write a brief explanation of machine learning for a 10-year-old.",
    "Give me a step-by-step guide to making scrambled eggs.",
    "List 3 ways to reduce your carbon footprint.",
    "Describe the solar system in one paragraph.",
    "Explain what a black hole is in simple terms.",
    "Give 3 reasons why reading is beneficial.",
    "What advice would you give someone learning to code?",
    "How can someone manage stress effectively?",
    "What are some ways to build self-confidence?",
]

print(f"Training prompts: {len(training_prompts)}")

## Part 5: Load Model + Reference Copy

f-GRPO requires a reference model (unlike standard GRPO with β=0). We load the base model in 4-bit, attach LoRA, and keep a frozen reference copy of the base for computing implicit rewards.

To fit both on a 24 GB GPU, we share the quantised base weights — only the LoRA parameters differ.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.print_trainable_parameters()
print(f"\nVRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## Part 6: f-GRPO Training Loop

Since TRL does not support f-GRPO, we implement a custom training loop:

```
for each prompt batch:
    1. Generate G completions per prompt (with the current policy)
    2. Score each completion with reward functions
    3. Compute group-normalised advantages
    4. Compute log-probs under current policy and reference (base with LoRA disabled)
    5. Compute f-GRPO loss using the chosen divergence
    6. Backpropagate and update LoRA parameters
```

For the reference model, we use `model.disable_adapter_layers()` to get $\pi_{\text{ref}}$ from the same model without loading a separate copy — this halves the VRAM cost.

In [ ]:
@torch.no_grad()
def generate_completions(model, tokenizer, prompt: str, num_completions: int = 2,
                         max_new_tokens: int = 128, temperature: float = 0.8):
    """Generate a group of completions for a single prompt."""
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    completions = []
    for _ in range(num_completions):
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        comp_ids = output[0][input_len:]
        comp_text = tokenizer.decode(comp_ids, skip_special_tokens=True)
        completions.append({"text": comp_text, "ids": comp_ids})

    return formatted, completions


def get_sequence_log_prob(model, tokenizer, prompt_text: str, completion_ids: torch.Tensor):
    """Compute the total log probability of a completion given a prompt."""
    prompt_ids = tokenizer(prompt_text, return_tensors="pt")["input_ids"].to(model.device)
    full_ids = torch.cat([prompt_ids, completion_ids.unsqueeze(0)], dim=1)

    with torch.no_grad() if not model.training else torch.enable_grad():
        outputs = model(full_ids)
        logits = outputs.logits

    # Get log probs for the completion tokens only
    prompt_len = prompt_ids.shape[1]
    # Shift: predict token i+1 from position i
    shift_logits = logits[0, prompt_len - 1:-1, :]  # (comp_len, vocab)
    shift_labels = completion_ids  # (comp_len,)

    log_probs = F.log_softmax(shift_logits.float(), dim=-1)
    token_log_probs = log_probs.gather(1, shift_labels.unsqueeze(1)).squeeze(1)

    return token_log_probs.sum()  # total log prob of sequence


def fgrpo_train_step(
    model, tokenizer, prompt: str, divergence: FDivergence,
    optimizer, num_completions: int = 2, max_new_tokens: int = 128,
    beta: float = 0.1,
):
    """One f-GRPO training step for a single prompt.

    Returns dict with loss, rewards, and advantages for logging.
    """
    # Step 1: Generate completions (no grad)
    model.eval()
    formatted_prompt, completions = generate_completions(
        model, tokenizer, prompt, num_completions, max_new_tokens
    )

    # Step 2: Score completions
    rewards = torch.tensor([
        compute_total_reward(prompt, c["text"]) for c in completions
    ], dtype=torch.float32)

    # Step 3: Group-normalise advantages
    if rewards.std() < 1e-8:
        # All rewards identical — no signal, skip
        return {"loss": 0.0, "rewards": rewards.tolist(), "advantages": [0.0] * len(completions), "skipped": True}

    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)

    # Step 4: Compute log probs under reference (LoRA disabled)
    model.eval()
    model.disable_adapter_layers()
    ref_log_probs = []
    for c in completions:
        ref_lp = get_sequence_log_prob(model, tokenizer, formatted_prompt, c["ids"])
        ref_log_probs.append(ref_lp.detach())
    ref_log_probs = torch.stack(ref_log_probs)
    model.enable_adapter_layers()

    # Step 5: Compute log probs under current policy (with grad)
    model.train()
    policy_log_probs = []
    for c in completions:
        policy_lp = get_sequence_log_prob(model, tokenizer, formatted_prompt, c["ids"])
        policy_log_probs.append(policy_lp)
    policy_log_probs = torch.stack(policy_log_probs)

    # Step 6: Compute f-GRPO loss
    advantages_device = advantages.to(policy_log_probs.device)
    loss = compute_fgrpo_loss(
        policy_log_probs, ref_log_probs, advantages_device,
        divergence, beta=beta,
    )

    # Step 7: Backprop
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return {
        "loss": loss.item(),
        "rewards": rewards.tolist(),
        "advantages": advantages.tolist(),
        "skipped": False,
    }


print("f-GRPO training functions defined.")

## Part 7: Train with KL Divergence

We run f-GRPO training using **KL divergence** as the default choice, for one epoch over our 30 prompts. This gives us a baseline to compare other divergences against.

In [ ]:
# Training hyperparameters
NUM_COMPLETIONS = 2      # Group size G (kept small for VRAM)
MAX_NEW_TOKENS = 128     # Max completion length
BETA = 0.1               # Implicit reward scaling
LEARNING_RATE = 5e-6
NUM_EPOCHS = 1

# Select divergence
divergence = KLDivergence()
print(f"Training with {divergence.name} divergence")
print(f"  Group size: {NUM_COMPLETIONS}")
print(f"  Beta: {BETA}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Prompts per epoch: {len(training_prompts)}")
print(f"  Epochs: {NUM_EPOCHS}")

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

# Training loop
history = defaultdict(list)
step = 0

print(f"\nStarting training...")
print(f"Peak VRAM before: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB\n")

for epoch in range(NUM_EPOCHS):
    # Shuffle prompts each epoch
    indices = np.random.permutation(len(training_prompts))

    for i, idx in enumerate(indices):
        prompt = training_prompts[idx]

        result = fgrpo_train_step(
            model, tokenizer, prompt, divergence, optimizer,
            num_completions=NUM_COMPLETIONS,
            max_new_tokens=MAX_NEW_TOKENS,
            beta=BETA,
        )

        step += 1
        history["step"].append(step)
        history["loss"].append(result["loss"])
        history["mean_reward"].append(np.mean(result["rewards"]))
        history["max_reward"].append(np.max(result["rewards"]))
        history["skipped"].append(result["skipped"])

        if step % 5 == 0 or step == 1:
            skip_tag = " [skipped]" if result["skipped"] else ""
            print(
                f"  Step {step:3d}/{len(training_prompts)*NUM_EPOCHS} | "
                f"Loss: {result['loss']:>7.4f} | "
                f"Rewards: {result['rewards']} | "
                f"VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB"
                f"{skip_tag}"
            )

        # Free VRAM periodically
        if step % 10 == 0:
            torch.cuda.empty_cache()

print(f"\nTraining complete!")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"Steps completed: {step}")
print(f"Steps skipped (no reward variance): {sum(history['skipped'])}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

non_skipped = [(s, l) for s, l, sk in zip(history["step"], history["loss"], history["skipped"]) if not sk]
if non_skipped:
    steps_ns, losses_ns = zip(*non_skipped)
    axes[0].plot(steps_ns, losses_ns, color="steelblue", alpha=0.5, label="Per-step")
    if len(losses_ns) >= 5:
        window = 5
        rolling = np.convolve(losses_ns, np.ones(window)/window, mode="valid")
        axes[0].plot(steps_ns[window-1:], rolling, color="darkblue", linewidth=2, label=f"{window}-step avg")
    axes[0].legend()
axes[0].set_xlabel("Step")
axes[0].set_ylabel("f-GRPO Loss")
axes[0].set_title(f"f-GRPO Training Loss ({divergence.name} Divergence)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["step"], history["mean_reward"], color="green", alpha=0.5, label="Mean")
axes[1].plot(history["step"], history["max_reward"], color="darkgreen", alpha=0.3, label="Max")
if len(history["mean_reward"]) >= 5:
    window = 5
    rolling = np.convolve(history["mean_reward"], np.ones(window)/window, mode="valid")
    axes[1].plot(history["step"][window-1:], rolling, color="darkgreen", linewidth=2, label=f"{window}-step avg")
axes[1].legend()
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Reward")
axes[1].set_title("Group Rewards Over Training")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 8: Save and Evaluate the KL f-GRPO Model

In [ ]:
# Save the KL f-GRPO adapter
model.save_pretrained("./results/fgrpo_kl/final")
tokenizer.save_pretrained("./results/fgrpo_kl/final")
print("KL f-GRPO adapter saved to ./results/fgrpo_kl/final")

# Generate test responses
test_prompts = [
    "What are the main causes of climate change?",
    "List 3 benefits of learning a second language.",
    "Explain how a computer virus works.",
    "What advice would you give to someone starting a small business?",
]

print("\n=== KL f-GRPO Model Responses ===")
model.eval()
kl_responses = {}

for prompt in test_prompts:
    _, completions = generate_completions(
        model, tokenizer, prompt, num_completions=1, max_new_tokens=200
    )
    response = completions[0]["text"]
    kl_responses[prompt] = response
    reward = compute_total_reward(prompt, response)

    print(f"\nPrompt: {prompt}")
    print(f"Response: {response[:300]}")
    print(f"Total reward: {reward:.2f}")
    print("-" * 60)

## Part 9: Compare Divergences

Now we compare different f-divergences by running the same training on each. To keep this tractable on a single GPU, we:
1. Reset LoRA weights before each run
2. Train for 1 epoch (30 steps) per divergence
3. Evaluate on the same test prompts

We compare **Reverse KL** (mode-seeking) and **Hellinger** (balanced) against our KL baseline.

In [ ]:
divergences_to_compare = [
    ("reverse_kl", ReverseKLDivergence()),
    ("hellinger", HellingerDivergence()),
]

all_histories = {"kl": dict(history)}  # Already have KL results
all_responses = {"kl": kl_responses}

for div_key, div in divergences_to_compare:
    print(f"\n{'='*60}")
    print(f"Training with {div.name} divergence")
    print(f"{'='*60}")

    # Reset LoRA weights
    for name, param in model.named_parameters():
        if "lora" in name.lower() and param.requires_grad:
            if "lora_A" in name:
                torch.nn.init.kaiming_uniform_(param)
            elif "lora_B" in name:
                torch.nn.init.zeros_(param)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )

    div_history = defaultdict(list)
    step = 0
    indices = np.random.permutation(len(training_prompts))

    for i, idx in enumerate(indices):
        prompt = training_prompts[idx]
        result = fgrpo_train_step(
            model, tokenizer, prompt, div, optimizer,
            num_completions=NUM_COMPLETIONS,
            max_new_tokens=MAX_NEW_TOKENS,
            beta=BETA,
        )
        step += 1
        div_history["step"].append(step)
        div_history["loss"].append(result["loss"])
        div_history["mean_reward"].append(np.mean(result["rewards"]))
        div_history["skipped"].append(result["skipped"])

        if step % 10 == 0 or step == 1:
            print(
                f"  Step {step:3d}/{len(training_prompts)} | "
                f"Loss: {result['loss']:>7.4f} | "
                f"Mean reward: {np.mean(result['rewards']):.2f}"
            )

        if step % 10 == 0:
            torch.cuda.empty_cache()

    all_histories[div_key] = dict(div_history)

    # Evaluate
    model.eval()
    div_responses = {}
    for prompt in test_prompts:
        _, completions = generate_completions(
            model, tokenizer, prompt, num_completions=1, max_new_tokens=200
        )
        div_responses[prompt] = completions[0]["text"]
    all_responses[div_key] = div_responses

    # Save
    model.save_pretrained(f"./results/fgrpo_{div_key}/final")
    print(f"  Saved to ./results/fgrpo_{div_key}/final")

print("\nAll divergence runs complete!")

## Part 10: Comparison Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
div_colors = {"kl": "#4C72B0", "reverse_kl": "#C44E52", "hellinger": "#55A868"}
div_labels = {"kl": "KL", "reverse_kl": "Reverse KL", "hellinger": "Hellinger"}

# Loss curves
for key, hist in all_histories.items():
    non_skipped = [(s, l) for s, l, sk in zip(hist["step"], hist["loss"], hist["skipped"]) if not sk]
    if non_skipped:
        steps_ns, losses_ns = zip(*non_skipped)
        if len(losses_ns) >= 5:
            window = 5
            rolling = np.convolve(losses_ns, np.ones(window)/window, mode="valid")
            axes[0].plot(steps_ns[window-1:], rolling, color=div_colors[key],
                        linewidth=2, label=f"{div_labels[key]} ({window}-step avg)")
        else:
            axes[0].plot(steps_ns, losses_ns, color=div_colors[key],
                        linewidth=2, label=div_labels[key])

axes[0].set_xlabel("Step")
axes[0].set_ylabel("f-GRPO Loss")
axes[0].set_title("Training Loss by Divergence")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reward curves
for key, hist in all_histories.items():
    if len(hist["mean_reward"]) >= 5:
        window = 5
        rolling = np.convolve(hist["mean_reward"], np.ones(window)/window, mode="valid")
        axes[1].plot(hist["step"][window-1:], rolling, color=div_colors[key],
                    linewidth=2, label=f"{div_labels[key]} ({window}-step avg)")
    else:
        axes[1].plot(hist["step"], hist["mean_reward"], color=div_colors[key],
                    linewidth=2, label=div_labels[key])

axes[1].set_xlabel("Step")
axes[1].set_ylabel("Mean Reward")
axes[1].set_title("Mean Reward by Divergence")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare responses and reward scores across divergences
print("=" * 90)
print("RESPONSE COMPARISON ACROSS DIVERGENCES")
print("=" * 90)

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print("-" * 90)

    for div_key in ["kl", "reverse_kl", "hellinger"]:
        resp = all_responses[div_key].get(prompt, "N/A")
        reward = compute_total_reward(prompt, resp)
        word_count = len(resp.split())
        print(f"\n  [{div_labels[div_key]:>12s}] (reward={reward:.2f}, words={word_count})")
        print(f"  {resp[:250]}")

    print()

In [ ]:
# Aggregate reward comparison
print(f"{'Divergence':<15} {'Format':>8} {'Concise':>8} {'NoRepeat':>8} {'Helpful':>8} {'Total':>8}")
print("-" * 65)

summary_data = {}
for div_key in ["kl", "reverse_kl", "hellinger"]:
    scores = {"format": [], "concise": [], "no_repeat": [], "helpful": []}
    for prompt in test_prompts:
        resp = all_responses[div_key].get(prompt, "")
        scores["format"].append(format_reward(resp))
        scores["concise"].append(conciseness_reward(resp))
        scores["no_repeat"].append(no_repetition_reward(resp))
        scores["helpful"].append(helpfulness_reward(prompt, resp))

    means = {k: np.mean(v) for k, v in scores.items()}
    total = sum(means.values())
    summary_data[div_key] = {**means, "total": total}

    print(f"{div_labels[div_key]:<15} {means['format']:>8.3f} {means['concise']:>8.3f} "
          f"{means['no_repeat']:>8.3f} {means['helpful']:>8.3f} {total:>8.3f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ["format", "concise", "no_repeat", "helpful"]
x = np.arange(len(metrics))
width = 0.25

for i, div_key in enumerate(["kl", "reverse_kl", "hellinger"]):
    vals = [summary_data[div_key][m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=div_labels[div_key],
           color=div_colors[div_key], edgecolor="black", linewidth=0.5)

ax.set_xlabel("Reward Function")
ax.set_ylabel("Mean Score")
ax.set_title("Reward Scores by f-Divergence Type")
ax.set_xticks(x + width)
ax.set_xticklabels([m.replace("_", " ").title() for m in metrics])
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Part 11: Gradient Landscape Visualisation

To build intuition about *why* different divergences produce different behaviours, let us visualise how the effective gradient magnitude varies with the implicit reward $r$ for positive and negative advantage completions.

In [ ]:
r = torch.linspace(-2, 2, 200, requires_grad=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(DIVERGENCES)))

for (key, cls), color in zip(DIVERGENCES.items(), colors):
    div = cls()

    # For positive advantage: gradient of g(r) w.r.t. r
    r_pos = r.clone().requires_grad_(True)
    g_r = div.link(r_pos)
    grad_pos = torch.autograd.grad(g_r.sum(), r_pos, create_graph=False)[0]

    # For negative advantage: gradient of f*(g(r)) w.r.t. r
    r_neg = r.clone().requires_grad_(True)
    g_r_neg = div.link(r_neg)
    fstar = div.f_conjugate(g_r_neg)
    fstar = torch.clamp(fstar, -10, 10)
    grad_neg = torch.autograd.grad(fstar.sum(), r_neg, create_graph=False)[0]

    axes[0].plot(r.numpy(), grad_pos.detach().numpy(), label=div.name, color=color, linewidth=2)
    axes[1].plot(r.numpy(), grad_neg.detach().numpy(), label=div.name, color=color, linewidth=2)

axes[0].set_xlabel("Implicit reward r")
axes[0].set_ylabel("dg(r)/dr")
axes[0].set_title("Gradient for Positive-Advantage Samples")
axes[0].set_ylim(-2, 3)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='black', linewidth=0.5)

axes[1].set_xlabel("Implicit reward r")
axes[1].set_ylabel("df*(g(r))/dr")
axes[1].set_title("Gradient for Negative-Advantage Samples")
axes[1].set_ylim(-3, 2)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='black', linewidth=0.5)

plt.suptitle("f-GRPO Effective Gradient by Divergence Type", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Observations:")
print("• KL/Pearson: constant gradient for D+ — equally aggressive at all reward levels")
print("• Reverse KL: large gradient for D+ when r is low — strongly pulls up weak completions")
print("• Total Variation: gradient → 0 as |r| grows — self-limiting, most stable")
print("• Hellinger/JS: moderate, decaying gradients — balanced exploration vs exploitation")

In [ ]:
# Cleanup
del model, optimizer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## Summary

### What We Covered

1. **f-Divergences** — Six different divergence families (KL, Reverse KL, Pearson, Hellinger, Jensen-Shannon, Total Variation) and their mathematical properties.

2. **f-GRPO algorithm** — How replacing GRPO's clipped-ratio objective with f-divergence variational bounds provides theoretical convergence guarantees and different optimisation dynamics.

3. **Custom implementation** — Built f-GRPO from scratch since TRL does not support it, using `disable_adapter_layers()` to efficiently share weights between the policy and reference model.

4. **Divergence comparison** — Trained and evaluated KL, Reverse KL, and Hellinger variants to see how the choice of divergence affects response quality.

5. **Gradient landscape** — Visualised how each divergence shapes the effective gradient for positive and negative advantage completions.

### Key Takeaways

| Divergence | Behaviour | Best For |
|---|---|---|
| **KL** | Mode-covering, constant gradients | General-purpose, exploration |
| **Reverse KL** | Mode-seeking, strong pull on low-reward | Focused, high-quality outputs |
| **Hellinger** | Balanced, bounded | Stable training, moderate exploration |
| **Jensen-Shannon** | Symmetric, bounded | When both over/under-coverage matter |
| **Total Variation** | Self-limiting, most conservative | Maximum stability, cautious alignment |
| **Pearson** | Aggressive on outliers | When extreme deviations are costly |

### Practical Recommendations

- **Start with KL** — it is the most studied and well-understood
- **Switch to Reverse KL** if you want sharper, more focused responses
- **Use Hellinger** if training is unstable — its bounded gradients prevent catastrophic updates
- **Consider Total Variation** for safety-critical applications where conservative updates are preferred

### References

- Haldar et al., *"f-GRPO and Beyond"*, arXiv 2602.05946, Feb 2026
- Shao et al., *"DeepSeekMath"*, arXiv 2402.03300, 2024
- DeepSeek AI, *"DeepSeek-R1"*, arXiv 2501.12948, 2025
- Nowozin et al., *"f-GAN"*, NeurIPS 2016 (foundational f-divergence variational estimation)